**ISD-1020 · Master ISD (apprenticeship) · Université Paris-Saclay**

**Before any edit:** `File → Save a copy in Drive`.
Work only on **your** copy.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, torch

print("sklearn", sklearn.__version__, "| torch", torch.__version__)

# Course helpers — skip this

This cell is **not** an exercise: it downloads the plotting helpers and dataset
loaders if they are not already next to the notebook. You do not need to read
it. Continue below.


In [ ]:
# Environment setup (when run outside the repository, e.g. Google Colab)
import sys
import urllib.request
from pathlib import Path

_RAW_BASE = "https://raw.githubusercontent.com/stephane-rivaud/M2-ISD-ML-DL/main"

for _start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_start / "pyproject.toml").is_file() and (_start / "common").is_dir():
        if str(_start) not in sys.path:
            sys.path.insert(0, str(_start))
        break

try:
    for module_path in ("common/__init__.py", "common/plots.py", "data/fetch.py"):
        local_file = Path(module_path)
        if local_file.is_file() or any(
            (Path(p) / module_path).is_file() for p in sys.path if p
        ):
            continue
        local_file.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{_RAW_BASE}/{module_path}", local_file)
except Exception:
    print(
        "Could not download the course helpers. Clone or download this repository so common/ and data/ sit next to this notebook."
    )


# First MLP in PyTorch — will the machine fail?

ISD-1020 · Monday 21 September 2026 · afternoon workshop (~140 min
after the 40 min lecture, including ~12 min of optional fast track)
· PUIO B210. Material: Stéphane Rivaud.

Thread of the module: **1. Problem · 2. Data · 3. Baseline · 4. Model · 5. Ablation ·
6. Error analysis → Recommendation**.
This morning: table, leakages, boosting. This afternoon: **first network**.
No prior deep-learning experience is assumed. Colab CPU
only — no GPU.

Honest landing point: on *this* small table, boosting still
wins. The goal is not to beat it, it is to **know how to build,
train and diagnose** a network.

In [ ]:
import random

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from common.plots import plot_learning_curves
from data.fetch import load_ai4i

SEED = 0
BATCH_SIZE = 256
LR = 1e-2
WIDTH = 32
EPOCHS = 40
# Small on purpose: four trainings, under 6 min on Colab CPU.
WIDTHS = (8, 64)
EPOCHS_GRID = (8, 50)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("seed", SEED)

## The problem — ⏱ ~8 min

Same dataset as this morning: **AI4I 2020** (predictive maintenance,
automotive / industry). One row = one machining cycle. The target
`Machine failure` is 1 if the machine fails.

**Decision.** Schedule maintenance, or not. A false negative = an
unanticipated failure. A false positive = a useless intervention.

This morning fixed the metrics (recall, PR-AUC, not accuracy) and
showed the **leakage** if we keep the sub-flags `TWF`, `HDF`, `PWF`,
`OSF`, `RNF`. We drop them. We also drop `UDI` and `Product ID`
(identifiers).

In pairs (30 s): if the network says "failure" too often, what does
that cost the shop floor?

## Data — ⏱ ~8 min

Local copy if it is there (`data/ai4i.parquet`), otherwise UCI. 10,000 rows, 14
columns. `Type` is product quality (L / M / H), not an identifier.

In [ ]:
df = load_ai4i()
TARGET = "Machine failure"
LEAKAGE = ["TWF", "HDF", "PWF", "OSF", "RNF"]
ID_COLS = ["UDI", "Product ID"]

print("shape:", df.shape)
print("columns:", list(df.columns))
print()
print(df[TARGET].value_counts())
print(f"failure rate = {df[TARGET].mean():.3%}")
print("Type:", df["Type"].value_counts().to_dict())
print("sub-flags (sum):", {c: int(df[c].sum()) for c in LEAKAGE})
df.head()

In [ ]:
feature_cols = [c for c in df.columns if c not in ID_COLS + LEAKAGE + [TARGET]]
X = df[feature_cols].copy()
y = df[TARGET].to_numpy(dtype=np.int64)
numeric_cols = [c for c in feature_cols if c != "Type"]
print("features:", feature_cols)
print("numeric:", numeric_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=SEED
)
print("train / val / test:", len(y_train), len(y_val), len(y_test))
print("rate train/val/test:", y_train.mean().round(4), y_val.mean().round(4), y_test.mean().round(4))
print(f"test failures = {int(y_test.sum())} / {len(y_test)}")

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric_cols),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["Type"],
        ),
    ]
)
X_train_np = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_np = preprocessor.transform(X_val).astype(np.float32)
X_test_np = preprocessor.transform(X_test).astype(np.float32)
N_FEATURES = X_train_np.shape[1]
print("n_features after encoding:", N_FEATURES)

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)
print(f"pos_weight = {n_neg} / {n_pos} = {float(pos_weight):.3f}")
print(f"majority accuracy (test) = {1.0 - y_test.mean():.3f}  |  no-skill PR-AUC ≈ {y_test.mean():.3f}")

Effective **stratified** 60 / 20 / 20 split (80% then 20% of train →
val). The `StandardScaler` and the encoder are fitted on the **inner
train only** — no leakage. `pos_weight` $n_{-}/n_{+}$ on this
same train: the loss pays more for a missed failure.

The width × epochs grid below is **small** (2 × 2): the four
trainings should finish in **under 6 min** on Colab CPU. If they
take much longer, something is wrong.

## Tensors in ten lines — ⏱ ~8 min

A PyTorch tensor is a NumPy array that can live on CPU or GPU
and, if needed, **retain the graph** for backpropagation.
Today: CPU, `float32`.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("a =", a)
print("shape", tuple(a.shape), "| dtype", a.dtype)
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
print("zeros + ones =\n", zeros + ones)
print("a @ a.T =\n", a @ a.T)
x_np = np.array([0.5, 1.5, 2.5], dtype=np.float32)
x_t = torch.from_numpy(x_np)
print("from numpy:", x_t, "→ back", x_t.numpy())

## The micrograd moment — autograd — ⏱ ~12 min

Let $y = w x + b$. By hand: $\partial y/\partial w = x$ and
$\partial y/\partial b = 1$. PyTorch does the same computation if $w$ and $b$
have `requires_grad=True`.

**Exercise.** Take $x=2$, $w=3$, $b=1$ (tensors). Only $w$ and $b$
request a gradient. Compute `y = w * x + b`, call `y.backward()`,
print `w.grad` and `b.grad`. Check against 2 and 1
(`torch.allclose`).

In [ ]:
# To complete
...

That is all of autograd: a graph of tensors, a `.backward()`, the
derivatives land on `.grad`. An MLP is only a longer **composition**
— the chain rule, automated. (The pedagogical idea comes
from *micrograd* (Karpathy, MIT); here it is PyTorch's autograd.)

## An MLP in `nn.Sequential` — ⏱ ~10 min

Features already standardised: 5 numeric + 3 `Type` indicators
= **8** inputs. Output: **one logit** (no sigmoid — the
`BCEWithLogitsLoss` does it, more stably).

Architecture: two hidden ReLU layers of width `width`, then a
`Linear(..., 1)`. `Dropout(p)` after each ReLU; `p=0` = no
dropout.

**Exercise.** Write `build_mlp(n_in, width, dropout=0.0)` that
returns an `nn.Sequential` in exactly this order:
`Linear → ReLU → Dropout → Linear → ReLU → Dropout → Linear`.

In [ ]:
# To complete
...

## Training loop — ⏱ ~15 min

A `DataLoader` serves **mini-batches**. The loss is
`BCEWithLogitsLoss(pos_weight=...)`: without this weight, the network learns
to always say "no failure" (96.6% accuracy, 0 recall).

At each batch: `zero_grad` → forward → loss → `backward` → `step`.
One epoch = one pass over the train set. We record the **mean** train
and val loss (val: `model.eval()`, no dropout).

In [ ]:
def make_loader(X_np, y_np, shuffle):
    """Tensor DataLoader; shuffle order is seeded when ``shuffle`` is True."""
    dataset = TensorDataset(
        torch.from_numpy(X_np),
        torch.from_numpy(np.asarray(y_np, dtype=np.float32)),
    )
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=shuffle, generator=generator
    )


@torch.no_grad()
def mean_loss(model, loader, criterion):
    """Dataset-mean loss with the model in eval mode."""
    model.eval()
    total = 0.0
    n = 0
    for xb, yb in loader:
        loss = criterion(model(xb).squeeze(-1), yb)
        total += loss.item() * yb.size(0)
        n += yb.size(0)
    return total / n


@torch.no_grad()
def evaluate(model, X_np, y_np, threshold=0.5):
    """PR-AUC, recall, precision, F1 on a numpy split (threshold on sigmoid)."""
    model.eval()
    logits = model(torch.from_numpy(X_np)).squeeze(-1)
    proba = torch.sigmoid(logits).cpu().numpy()
    pred = (proba >= threshold).astype(np.int64)
    y_np = np.asarray(y_np)
    return {
        "pr_auc": float(average_precision_score(y_np, proba)),
        "recall": float(recall_score(y_np, pred)),
        "precision": float(precision_score(y_np, pred, zero_division=0)),
        "f1": float(f1_score(y_np, pred)),
    }


def print_metrics(title, metrics):
    print(title)
    for key, value in metrics.items():
        print(f"  {key:10s} {value:.3f}")

**Exercise.** Write `train_mlp(model, epochs, lr=LR, weight_decay=0.0,
patience=None)`:

1. Recreate the loaders (`make_loader` on train / val) **inside** the
   function, so each call restarts from the same shuffle (`SEED`).
2. `Adam` on `model.parameters()`, `BCEWithLogitsLoss(pos_weight=pos_weight)`.
3. Epoch loop: `model.train()`; for each batch, the four
   calls `zero_grad` / forward / `backward` / `step`; accumulate the
   mean train loss.
4. Append the val loss via `mean_loss`.
5. If `patience` is an integer: remember the best `state_dict` on
   val, stop after `patience` epochs without a gain ($>10^{-4}$),
   restore the best weights.
6. Return `(train_losses, val_losses)`.

Logits are `(batch, 1)`: `squeeze(-1)` them before the loss.

In [ ]:
# To complete
...

## Learning curves — ⏱ ~8 min

A `width=32` network, 40 epochs, `lr=1e-2`, no regularisation.
We read the curve **together** with the test metrics (threshold 0.5 on the
sigmoid). PR-AUC does not depend on the threshold; recall does.

In [ ]:
torch.manual_seed(SEED)
model = build_mlp(N_FEATURES, width=WIDTH)
train_losses, val_losses = train_mlp(model, epochs=EPOCHS)
plot_learning_curves(train_losses, val_losses)
plt.show()
print(f"last train loss = {train_losses[-1]:.3f} | last val loss = {val_losses[-1]:.3f}")
print_metrics("MLP  (width=32, 40 epochs, test)", evaluate(model, X_test_np, y_test))

## Width × epochs: under- and overfitting — ⏱ ~12 min

Four trainings only: widths 8 and 64, budgets 8 and 50
epochs. Small on purpose — expect under 6 min on Colab CPU.

**Watch for.** Narrow network + few epochs: train **and** val loss
high (underfitting). Wide network + many epochs: train
loss clearly below val (overfitting).

**Exercise.** Nested loop over `WIDTHS` and `EPOCHS_GRID`. At each
pair: `torch.manual_seed(SEED)`, `build_mlp`, `train_mlp`,
`evaluate` on the test set. Fill a `DataFrame` with `width`,
`epochs`, last train loss, last val loss, `pr_auc`,
`recall`.

In [ ]:
# To complete
...

Typical reading (check against *your* table): (8, 8) underfits;
(64, 50) opens a train/val gap. The best PR-AUC is not
necessarily the run with the lowest train loss.

## Regularisation — ⏱ ~15 min

Three levers, **same** `width=32` architecture and **same** 40-epoch
budget as the model above. We compare with the unregularised run
already in memory (`model`, `train_losses`, `val_losses`).

**Exercise — weight decay.** Rerun an MLP `width=32` with
`weight_decay=1e-2` (L2 penalty in Adam). Print the last
losses and the test metrics.

In [ ]:
# To complete
...

**Exercise — dropout.** Same thing with `dropout=0.3` (and
`weight_decay=0`). Dropout acts only in `model.train()`; val
goes through `eval`.

In [ ]:
# To complete
...

**Exercise — early stopping.** `patience=8`: we stop if val no longer
improves, and we **restore** the best weights. Print the
number of epochs actually run (`len(train_es)`) and the
test metrics.

In [ ]:
# To complete
...

## MLP versus boosting — same split — ⏱ ~15 min

`HistGradientBoostingClassifier` on **the same** 8 standardised
features, **the same** 6,400 train rows, **the same** test set.
That is the honest comparison, not "HGB with more data".

The PR-AUC obtained here is therefore a little **below** this morning's (0.805):
this morning's HGB saw `Type` and the full set of columns. We do not compare
the two numbers with each other; we compare MLP and HGB **in this
notebook**.

In [ ]:
hgb = HistGradientBoostingClassifier(max_iter=200, random_state=SEED)
hgb.fit(X_train_np, y_train)
hgb_scores = hgb.predict_proba(X_test_np)[:, 1]
hgb_pred = hgb.predict(X_test_np)
hgb_metrics = {
    "pr_auc": float(average_precision_score(y_test, hgb_scores)),
    "recall": float(recall_score(y_test, hgb_pred)),
    "precision": float(precision_score(y_test, hgb_pred, zero_division=0)),
    "f1": float(f1_score(y_test, hgb_pred)),
}
mlp_metrics = evaluate(model, X_test_np, y_test)
print_metrics("MLP   (width=32, 40 epochs, threshold 0.5)", mlp_metrics)
print_metrics("HGB   (max_iter=200, same split)", hgb_metrics)

**Take-away for the oral exam.** On AI4I (10k rows, 8 features),
**boosting wins PR-AUC** without substantial work on the network.
The MLP, with `pos_weight`, often has **higher recall** and
lower precision: it cries failure more often. That is not
a lab failure. The pedagogical point: you now know how to
**build, train and diagnose** a network — not that it beats
HGB on a small table.

A network starts to pay on tables when there is much more
data, mixed modalities, embeddings, or a real architecture
budget. Not here, not tonight.

## Fast track — ⏱ ~12 min (optional)

For those who have already done ML: a learning-rate sweep,
then the variance due to `torch.manual_seed`.

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
print("learning-rate sweep (width=32, 25 epochs)")
for lr in (1e-3, 1e-2, 1e-1):
    torch.manual_seed(SEED)
    model_lr = build_mlp(N_FEATURES, width=WIDTH)
    train_lr, val_lr = train_mlp(model_lr, epochs=25, lr=lr)
    metrics_lr = evaluate(model_lr, X_test_np, y_test)
    print(
        f"  lr={lr:.0e}  train={train_lr[-1]:.3f}  val={val_lr[-1]:.3f}  "
        f"PR-AUC={metrics_lr['pr_auc']:.3f}  recall={metrics_lr['recall']:.3f}"
    )

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
print("seed variance (width=32, 25 epochs, lr=1e-2)")
seed_rows = []
for seed in (0, 1, 2):
    torch.manual_seed(seed)
    model_seed = build_mlp(N_FEATURES, width=WIDTH)
    train_mlp(model_seed, epochs=25)
    metrics_seed = evaluate(model_seed, X_test_np, y_test)
    seed_rows.append({"seed": seed, **metrics_seed})
    print(
        f"  seed={seed}  PR-AUC={metrics_seed['pr_auc']:.3f}  "
        f"recall={metrics_seed['recall']:.3f}"
    )
print(pd.DataFrame(seed_rows).round(3).to_string(index=False))

## Attributions

- **AI4I 2020 Predictive Maintenance**: UCI ML repository, dataset
  [601](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset),
  licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- **Autograd / tensors / `nn.Sequential` / loop**: official
  PyTorch tutorials (*Deep Learning with PyTorch: A 60 Minute Blitz* —
  autograd, neural networks), licence
  [BSD-3-Clause](https://github.com/pytorch/tutorials).
- **MLP, regularisation, curves**: [d2l.ai](https://d2l.ai) chapters
  4–5 (*Multilayer Perceptrons*), licence
  [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
- **Scalar autograd moment**: inspired by
  [micrograd](https://github.com/karpathy/micrograd) (Andrej Karpathy),
  MIT licence — the implementation here is PyTorch's.

## To go further at home

1. *Deep Learning with PyTorch: A 60 Minute Blitz* — autograd then
   `nn`:
   [tutorial](https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html)
2. d2l.ai, chapters 4 and 5 — MLP, dropout, weight decay, early stopping:
   [Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/index.html)

## Empirical protocol — ⏱ ~10 min

The six cells below are the thread of the module (and of the oral exam).
Answer in prose, tied to *this* notebook.

## Problem

Answer in prose (not only code).

1. Which **business decision** should the model inform, and what is the **target**?
2. Which **metric** matches that decision, and why not accuracy alone?
3. How do you separate training and test **without leakage** (split unit, time, stratification)?

## Data

Answer in prose (not only code).

1. What are the variables, their types, and any notable **imbalances** or volumes?
2. Which columns must you **not** use (identifiers, leakages, target sub-flags)?
3. What do you know about **quality** (missing values, outliers, drift) and temporal or business coverage?

## Baseline

Answer in prose (not only code).

1. Which **naive baseline** (majority class, mean, seasonal) and what score does it get?
2. Why is this baseline the **honest floor**, and not a "deliberately weak" model?
3. Which score must you **beat** to justify a more complex model?

## Model

Answer in prose (not only code).

1. Which **model family** do you choose, and why (not "because it is deep")?
2. How do you train it (split, validation, hyperparameters, loss)?
3. Does the model **beat the baseline** on the chosen metric, on an uncontaminated split?

## Ablation

Answer in prose (not only code).

1. What happens if you remove a family of variables, a regulariser, or a block of the network?
2. Which choice (features, architecture, horizon, threshold) **actually changes** the score?
3. Does the gain justify the extra **complexity** relative to the baseline?

## Error analysis → Recommendation

Answer in prose (not only code).

1. Where does the model go wrong (**segments**, error types, horizon)?
2. Are these errors **costly** for the business, and what would you change in the data or the model?
3. What concrete **recommendation**: deploy, do not deploy, stay on the baseline, or collect this specific data?